In [4]:
import time
import random
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import pandas as pd
from langchain_upstage import ChatUpstage
from sqlalchemy import text
from dotenv import load_dotenv, find_dotenv
from web.config.database import SessionLocal

load_dotenv(find_dotenv())

# ----------------------------------------------------------------
# 2. LLM and Prompt Setup (Modern LCEL Style)
# ----------------------------------------------------------------

# Chat 모델 초기화
llm = ChatUpstage()
output_parser = StrOutputParser()

# --- [설정] 텍스트 분할기 (Map-Reduce용) ---
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=20000,
    chunk_overlap=1000,
    separators=["\n\n", "\n", " ", ""]
)

# --- [Map 단계] 부분 요약 프롬프트 ---
map_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 긴 문서에서 핵심 사실을 빠짐없이 추출하는 데이터 분석가입니다."),
    ("human", """
    다음은 금융 공시 문서의 일부입니다.
    이후 단계에서 초보자를 위한 쉬운 글을 작성할 수 있도록,
    주어진 텍스트 이외의 내용을 요약해서는 안됩니다.
    중요한 사실(숫자, 핵심 기술, 사업 모델, 시장 상황) 위주로 내용을 요약해주세요.

    [문서 일부]
    {text}

    요약:
    """)
])

map_chain = map_prompt | llm | output_parser

# --- [Reduce 단계] 1. 회사 개요 최종 요약 ---
overview_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 어려운 금융 용어를 주식 초보자도 이해하기 쉽게 풀어주는 '친절한 투자 멘토'입니다."),
    ("human", """
    다음은 회사의 개요에 대한 핵심 내용들입니다.
    이 내용을 바탕으로, 주식을 처음 시작하는 사람도 이 회사가 **'도대체 무엇으로 돈을 버는 회사인지'** 단번에 이해할 수 있도록 1000자 이내의 쉬운 줄글로 설명해주세요.
    주어진 텍스트 이외의 내용을 요약해서는 안됩니다.

    [필수 작성 지침]
    1. **전문 용어 금지**: '영업수익', '당기순이익' 같은 단어 대신 '매출', '순수익' 처럼 쉬운 말로 바꾸거나 풀어서 설명하세요.
    2. **비유와 예시 활용**: 비즈니스 모델이 어렵다면 일상생활의 예시를 들어 설명하세요.
    3. **스토리텔링**: 딱딱한 보고서체가 아니라, 옆에서 말해주는 듯한 부드러운 어조(해요체 또는 부드러운 서술형)를 사용하세요.
    4. **형식**: 절대 번호나 글머리 기호(1., -)를 쓰지 말고, 자연스러운 문단으로 이어지게 작성하세요.

    [예시]
    삼성화재는 1952년에 설립된 국내 최대 규모의 손해보험 회사입니다. 주요 사업은 보험업법에서 규정한 손해보험과 제3보험업을 기본으로 하고 있어요. 쉽게 말해, 다양한 위험으로부터 고객들을 보호하는 보험을 판매하는 일을 하는 거죠. 예를 들어, 자동차 사고나 건강 문제, 화재 같은 위험에 대비한 보험을 제공하고 있어요.
    보험 판매뿐 아니라, 관계 법령에 따라 개인연금, 퇴직연금, 신탁 같은 금융상품도 판매하고 있습니다. 또한, 회사는 모은 자금을 투자로 운용하여 수익을 창출하고 있어요. 회사가 제공하는 상품 중 일반보험, 장기보험, 자동차보험의 비중은 보험수익 기준으로 각각 14.7%, 50.9%, 34.4%입니다.
    삼성화재는 국내외에 여러 종속회사를 운영하고 있어요. 국내에는 손해사정 전문회사, 보험 상담 및 손해사정 서비스를 제공하는 회사, 손해보험 상품과 제휴 생명보험 상품을 제공하는 판매회사 등이 있습니다. 해외에는 인도네시아, 베트남, 유럽 등 3개국에 자회사를 두고, 각 국가에서 한국계 기업을 중심으로 보험 서비스를 제공하고 있어요.
    회사는 글로벌 신용평가사인 Standard & Poor's와 A.M.Best로부터 최고 수준의 신용등급을 지속적으로 받고 있어, 재무적으로 매우 안정적이라는 평가를 받고 있습니다. 이러한 신용등급은 회사가 고객에게 보험금을 안정적으로 지급할 수 있는 능력을 보여줍니다.
    삼성화재는 다양한 상을 수상하며 고객 만족도에서도 높은 평가를 받고 있어요. 예를 들어, 한국서비스품질지수, 한국산업의 고객만족도, 국가고객만족도 등에서 여러 해 연속 1위를 차지했어요. 이러한 성과는 회사가 고객에게 최상의 서비스를 제공하기 위해 노력하고 있음을 보여줍니다.
    종합적으로, 삼성화재는 다양한 보험 상품과 금융 서비스를 제공하며 고객의 위험을 관리하고, 안정적인 재무상태와 높은 고객 만족도를 자랑하는 회사입니다.

    [내용]
    {text}
    """)
])

description_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 복잡한 기술과 산업 이야기를 대중에게 쉽게 전달하는 'IT/산업 전문 칼럼니스트'입니다."),
    ("human", """
    다음은 회사의 상세 사업 내용들입니다.
    이 내용을 바탕으로 이 회사의 **주요 제품과 서비스가 우리 일상 생활 어디에 쓰이는지**, 그리고 **시장에서 어떤 경쟁력이 있는지** 1000자 이내의 쉬운 줄글로 설명해주세요.
    주어진 텍스트 이외의 내용을 요약해서는 안됩니다.

    [필수 작성 지침]
    1. **쉬운 설명**: 기술적인 스펙 나열보다는, 그 기술이 소비자에게 어떤 가치를 주는지(예: '편리함', '비용 절감')에 집중하세요.
    2. **독자 중심**: "이 회사는 B2B 사업을 영위함" 보다는 "이 회사는 다른 기업들에게 부품을 납품하여 돈을 법니다"와 같이 구체적인 상황을 그려주세요.
    3. **흐름 유지**: 각 사업부가 따로 노는 느낌이 아니라, 회사의 전체적인 사업 방향성이 보이도록 연결하여 작성하세요.
    4. **스토리텔링**: 딱딱한 보고서체가 아니라, 옆에서 말해주는 듯한 부드러운 어조(해요체 또는 부드러운 서술형)를 사용하세요.
    5. **형식**: 번호 매기기를 하지 말고, 한 편의 읽기 쉬운 수필이나 기사처럼 작성하세요.

    [예시]
    삼성화재해상보험은 우리의 일상 생활 곳곳에 스며들어 있는 회사입니다. 먼저, 우리가 자동차를 운전하거나 집을 소유할 때 드는 보험을 생각해보세요. 삼성화재해상보험은 국내 손해보험 시장에서 큰 비중을 차지하며, 자동차보험, 화재보험, 해상보험 등 다양한 보험상품을 제공합니다. 이를 통해 우리는 예상치 못한 사고나 재난으로부터 재정적 보호를 받을 수 있습니다.
    하지만 삼성화재해상보험의 활약은 국내에만 국한되지 않습니다. 세계 여러 나라에서도 그 영향력을 발휘하고 있습니다. 예를 들어, 유럽, 베트남, 싱가포르, 인도네시아 등에서 해외사업을 진행하며, 각 국가의 특성에 맞춰 보험 서비스를 제공하고 있습니다. 이를 가능하게 하는 것은 바로 글로벌 통합시스템입니다. 이 시스템을 통해 중앙집중화된 IT 환경에서 해외 거점의 운영 현황을 실시간으로 파악하고 관리할 수 있습니다.
    이 회사의 또 다른 강점은 해외사업 인력 양성에 있습니다. 해외사업 전문가를 양성하여 인적자원의 글로벌화를 추진하고 있습니다. 이를 통해 각 국가의 시장 특성을 잘 이해하고, 그에 맞는 보험 서비스를 제공할 수 있습니다. 또한, 해외투자 프로세스를 확립하여 신속한 내부 의사결정을 가능하게 하고 있습니다.
    특히 주목할 만한 점은 영국 로이즈 손보사인 캐노피우스에 대한 투자입니다. 이 투자를 통해 선진 보험시장의 경영 역량을 확보하고, 글로벌 손해보험 시장에서 경쟁력을 강화하고 있습니다. 최근 몇 년간 대형 자연재해, 전염병, 지정학적 리스크 등으로 글로벌 손해보험 시장의 시장가격이 상승세에 있어, 이러한 전략은 회사의 장기적인 성장에 큰 도움이 될 것입니다.
    중국 시장에 대해서도 삼성화재해상보험은 적극적입니다. 중국 최대 IT 기업인 텐센트와 협력하여 중국법인을 합작법인 형태로 전환하였습니다. 이를 통해 중국 시장의 전문가를 경영진으로 영입하고, 신규 거버넌스를 구성하여 중국 시장에서의 경쟁력을 높이고 있습니다.
    이 회사는 보험사업 외에도 손해사정서비스, 고객상담서비스, 출동서비스, 보험대리점 영업 등 다양한 기타 사업을 영위하고 있습니다. 손해사정서비스는 보험금 지급 여부를 판단하고 지급액을 결정하는 중요한 역할을 하며, 고객상담서비스는 고객의 문의를 신속하고 정확하게 처리하여 고객 만족도를 높입니다. 출동서비스는 사고 발생 시 빠르게 출동하여 고객의 불편을 최소화하고, 보험대리점 영업은 다양한 채널을 통해 보험상품을 판매하여 고객의 선택의 폭을 넓히고 있습니다.
    이러한 다양한 사업을 통해 삼성화재해상보험은 국내뿐만 아니라 해외 시장에서도 강력한 경쟁력을 발휘하고 있습니다. 2024년 반기 국내 손해보험시장 규모에서 주요 손보사 4개사 중 하나로 약 73%의 점유율을 차지하고 있으며, 해외에서도 꾸준히 사업을 확장하고 있습니다. 또한, 회사의 수익성 지표인 운용자산이익률, 영업이익률, 총자산수익률, 자기자본수익률 등이 모두 높은 수준을 유지하고 있어, 안정적인 재무구조를 가지고 있다는 것을 알 수 있습니다.

    [내용]
    {text}
    """)
])

# 최종 체인
overview_chain = overview_prompt | llm | output_parser
description_chain = description_prompt | llm | output_parser

# ----------------------------------------------------------------
# [수정된 부분] 3. Smart Summary Logic (내부 로직 변경)
# ----------------------------------------------------------------

def run_smart_summary(text_content, final_chain):
    """
    텍스트 길이에 따라 단순 요약 또는 맵-리듀스를 선택하며,
    API Rate Limit(429) 에러를 방지하기 위해 Retry 및 Throttling 로직을 포함합니다.
    """
    if not text_content or pd.isna(text_content):
        return ""

    # --- [내부 헬퍼 함수] 안전한 호출을 위한 Retry 로직 ---
    def invoke_with_retry(chain, input_data, max_retries=5):
        for attempt in range(max_retries):
            try:
                return chain.invoke(input_data)
            except Exception as e:
                error_msg = str(e)
                # 429(Rate Limit) 에러인 경우 대기 후 재시도
                if "429" in error_msg or "rate limit" in error_msg.lower():
                    wait_time = (2 ** attempt) + random.uniform(0, 1) # 지수 백오프
                    print(f"      ⚠️ Rate Limit 발생. {wait_time:.1f}초 대기 후 재시도 ({attempt+1}/{max_retries})...")
                    time.sleep(wait_time)
                else:
                    raise e # 다른 에러면 즉시 중단
        raise Exception("API 호출 실패: 최대 재시도 횟수 초과")

    # 1. 길이가 짧은 경우 (약 20,000자 미만)
    if len(text_content) < 20000:
        return invoke_with_retry(final_chain, {"text": text_content})

    # 2. 길이가 긴 경우 (Map-Reduce 실행)
    print(f"   ㄴ 텍스트가 깁니다({len(text_content)}자). Map-Reduce 분할 처리를 시작합니다.")

    # (1) Split
    docs = text_splitter.create_documents([text_content])
    split_texts = [doc.page_content for doc in docs]
    print(f"   ㄴ {len(split_texts)}개의 청크로 분할됨. 부분 요약 생성 중...")

    # (2) Map (Throttling 적용)
    # 한 번에 batch를 보내지 않고, 3개씩 끊어서 보내고 중간에 쉽니다.
    chunk_summaries = []
    batch_size = 3

    for i in range(0, len(split_texts), batch_size):
        # 미니 배치 생성
        current_batch_texts = split_texts[i : i + batch_size]

        # 현재 배치의 각 텍스트 처리
        for text_piece in current_batch_texts:
            res = invoke_with_retry(map_chain, {"text": text_piece})
            chunk_summaries.append(res)

        # 배치 사이 휴식 (API 호출 간격을 둠)
        if i + batch_size < len(split_texts):
            time.sleep(3)

    # (3) Reduce
    combined_summary = "\n\n".join(chunk_summaries)
    print("   ㄴ 부분 요약 완료. 최종 요약 생성 중...")

    return invoke_with_retry(final_chain, {"text": combined_summary})


# ----------------------------------------------------------------
# 4. Data Processing Functions
# ----------------------------------------------------------------

def get_all_company_ids(db_session):
    query = "SELECT id FROM company WHERE overview is null and description is null order by id asc"
    result = db_session.execute(text(query)).fetchall()
    return [row[0] for row in result]

def get_disclosure_data(db_session, company_id):
    query_overview = """
    SELECT company_overview
    FROM parsed_disclosure_file
    WHERE company_id = :company_id
      AND company_overview IS NOT NULL
    ORDER BY CHAR_LENGTH(company_overview) DESC
    LIMIT 1
    """

    query_business = """
    SELECT business_contents
    FROM parsed_disclosure_file
    WHERE company_id = :company_id
      AND business_contents IS NOT NULL
    ORDER BY CHAR_LENGTH(business_contents) DESC
    LIMIT 1
    """

    overview_row = db_session.execute(text(query_overview), {'company_id': company_id}).fetchone()
    business_row = db_session.execute(text(query_business), {'company_id': company_id}).fetchone()

    # 결과 추출 (데이터가 없을 경우 빈 문자열 처리)
    overview_text = overview_row[0] if overview_row else ""
    business_text = business_row[0] if business_row else ""

    # 하나의 DataFrame으로 병합
    return pd.DataFrame({
        'company_overview': [overview_text],
        'business_contents': [business_text]
    })

def update_company_summary(db_session, company_id, overview, description):
    query = """
    UPDATE company
    SET overview = :overview, description = :description
    WHERE id = :company_id
    """
    db_session.execute(text(query), {
        'overview': overview,
        'description': description,
        'company_id': company_id
    })
    db_session.commit()
    print(f"✅ Updated summary for company ID: {company_id}")

In [5]:
# [테스트 코드]
db_session = SessionLocal()
company_id = 85
print(f"Processing company ID: {company_id}")

try:
    # 1. Fetch data
    disclosure_df = get_disclosure_data(db_session, company_id)

    # 2. Summarize Company Overview
    overviews = disclosure_df['company_overview'].dropna().tolist()
    if overviews:
        # 여러 개요 중 가장 긴 것을 선택
        longest_overview = max(overviews, key=len)
        # [수정] 직접 invoke하지 않고 run_smart_summary를 사용해 긴 텍스트 대응
        summarized_overview = run_smart_summary(longest_overview, overview_chain)
    else:
        summarized_overview = ""

    # 3. Summarize Business Content
    business_contents = disclosure_df['business_contents'].dropna().tolist()
    if business_contents:
        # 여러 내용을 합쳐서 처리
        full_business_content = "\n\n".join(business_contents)
        # [수정] 직접 invoke하지 않고 run_smart_summary 사용
        summarized_description = run_smart_summary(full_business_content, description_chain)
    else:
        summarized_description = ""

    print("----------기업 개요 (결과)------------")
    print(summarized_overview)
    print("\n----------기업 설명 (결과)-------------")
    print(summarized_description)

finally:
    db_session.close()

Processing company ID: 85
   ㄴ 텍스트가 깁니다(46484자). Map-Reduce 분할 처리를 시작합니다.
   ㄴ 3개의 청크로 분할됨. 부분 요약 생성 중...
   ㄴ 부분 요약 완료. 최종 요약 생성 중...
----------기업 개요 (결과)------------
삼성화재는 1952년에 설립된 국내 최대 규모의 손해보험 회사입니다. 이 회사는 다양한 위험으로부터 고객들을 보호하는 보험을 판매하는 일을 하고 있어요. 예를 들어, 자동차 사고나 건강 문제, 화재 같은 위험에 대비한 보험을 제공하고 있죠. 보험 판매뿐 아니라, 회사는 모은 자금을 투자로 운용하여 수익을 창출하고 있습니다.

삼성화재는 국내와 해외에 여러 종속회사를 운영하고 있어요. 국내 종속회사들은 손해사정, 보험 상담 및 손해사정 서비스, 판매회사 등을 운영하고 있고, 해외 종속회사들은 인도네시아, 베트남, 유럽 등 3개국에서 한국계 기업을 중심으로 보험 서비스를 제공하고 있답니다.

회사는 글로벌 신용평가사인 Standard & Poor's와 A.M.Best로부터 최고 수준의 신용등급을 지속적으로 받고 있어, 재무적으로 매우 안정적이라는 평가를 받고 있어요. 또한, 다양한 상을 수상하며 고객 만족도에서도 높은 평가를 받고 있어요.

종합적으로, 삼성화재는 다양한 보험 상품과 금융 서비스를 제공하며 고객의 위험을 관리하고, 안정적인 재무상태와 높은 고객 만족도를 자랑하는 회사입니다.

----------기업 설명 (결과)-------------
삼성화재해상보험은 우리 일상생활과 떼려야 뗄 수 없는 회사입니다. 먼저, 자동차나 집을 소유하고 있다면, 이 회사가 제공하는 보험을 떠올려보세요. 삼성화재는 자동차보험, 화재보험, 해상보험 등 다양한 보험상품을 통해 우리가 예상치 못한 사고나 재난에서 재정적 보호를 받을 수 있도록 돕습니다.

하지만 이 회사의 활약은 국내에만 국한되지 않습니다. 유럽, 베트남, 싱가포르, 인도네시아 등 세계 여러 나라에서도 그 영향력을 

In [6]:
# [파이프라인 코드]

db_session = SessionLocal()
try:
    company_ids = get_all_company_ids(db_session)
    print(f"총 {len(company_ids)}개의 회사를 처리합니다.")

    for company_id in company_ids:
        print(f"\nProcessing company ID: {company_id}")

        # 1. Fetch data
        disclosure_df = get_disclosure_data(db_session, company_id)

        if disclosure_df.empty:
            print(f" -> No disclosure data found for company ID: {company_id}")
            continue

        # 2. Summarize Company Overview
        overviews = disclosure_df['company_overview'].dropna().tolist()
        if overviews:
            longest_overview = max(overviews, key=len)
            # [수정] Map-Reduce 자동 적용 함수 호출
            summarized_overview = run_smart_summary(longest_overview, overview_chain)
        else:
            summarized_overview = ""

        # 3. Summarize Business Content
        # [수정] 오타 수정: business_content -> business_contents (DB 컬럼명 일치)
        business_contents = disclosure_df['business_contents'].dropna().tolist()
        if business_contents:
            full_business_content = "\n\n".join(business_contents)
            # [수정] Map-Reduce 자동 적용 함수 호출
            summarized_description = run_smart_summary(full_business_content, description_chain)
        else:
            summarized_description = ""

        # 4. Update DB
        # 내용이 하나라도 있을 때만 업데이트 (선택 사항)
        if summarized_overview or summarized_description:
            update_company_summary(db_session, company_id, summarized_overview, summarized_description)
        else:
            print(" -> 요약할 내용이 없어 업데이트를 건너뜁니다.")

except Exception as e:
    print(f"CRITICAL ERROR in pipeline: {e}")
finally:
    db_session.close()

총 457개의 회사를 처리합니다.

Processing company ID: 108
   ㄴ 텍스트가 깁니다(36667자). Map-Reduce 분할 처리를 시작합니다.
   ㄴ 2개의 청크로 분할됨. 부분 요약 생성 중...
      ⚠️ Rate Limit 발생. 1.9초 대기 후 재시도 (1/5)...
      ⚠️ Rate Limit 발생. 2.1초 대기 후 재시도 (2/5)...
   ㄴ 부분 요약 완료. 최종 요약 생성 중...
      ⚠️ Rate Limit 발생. 1.6초 대기 후 재시도 (1/5)...
   ㄴ 텍스트가 깁니다(164965자). Map-Reduce 분할 처리를 시작합니다.
   ㄴ 9개의 청크로 분할됨. 부분 요약 생성 중...
      ⚠️ Rate Limit 발생. 1.1초 대기 후 재시도 (1/5)...
      ⚠️ Rate Limit 발생. 1.5초 대기 후 재시도 (1/5)...
      ⚠️ Rate Limit 발생. 2.1초 대기 후 재시도 (2/5)...
      ⚠️ Rate Limit 발생. 1.1초 대기 후 재시도 (1/5)...
      ⚠️ Rate Limit 발생. 2.4초 대기 후 재시도 (2/5)...
      ⚠️ Rate Limit 발생. 4.8초 대기 후 재시도 (3/5)...
      ⚠️ Rate Limit 발생. 1.0초 대기 후 재시도 (1/5)...
      ⚠️ Rate Limit 발생. 1.8초 대기 후 재시도 (1/5)...
      ⚠️ Rate Limit 발생. 1.6초 대기 후 재시도 (1/5)...
   ㄴ 부분 요약 완료. 최종 요약 생성 중...
✅ Updated summary for company ID: 108

Processing company ID: 119
   ㄴ 텍스트가 깁니다(29468자). Map-Reduce 분할 처리를 시작합니다.
   ㄴ 2개의 청크로 분할됨. 부분 요약 생성 중...
   ㄴ 부분 요약 완료. 최종 요약 생성 

KeyboardInterrupt: 